# 01 · Data Exploration

Load the first 10 K rows from each Alibaba cluster-trace table via DuckDB
(no full table ever enters Python memory).  For each table we print:
- Schema
- `.head(5)`
- `.describe()` (numeric columns)
- Null-rate summary
- Timestamp range (where applicable)

**Before running:** set `TRACE` to `"v2026"` or `"v2023"` and confirm
the raw files exist under `data/raw/{TRACE}/`.

In [ ]:
import duckdb
import polars as pl
from pathlib import Path
from IPython.display import display

from src.config import RAW_DIR, DUCKDB_MEMORY_LIMIT, DUCKDB_THREADS
from src.ingest.schema import SCHEMAS

TRACE = "v2026"   # ← change to "v2023" for the validation set
SAMPLE = 10_000   # rows to pull into memory for exploration

RAW = RAW_DIR / TRACE
con = duckdb.connect()
con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
con.execute(f"SET threads={DUCKDB_THREADS}")
print(f"DuckDB {duckdb.__version__} ready — scanning {RAW}")

In [ ]:
def explore_table(table_name: str) -> None:
    path = RAW / f"{table_name}.parquet"
    if not path.exists():
        print(f"  SKIP — {path} not found")
        return

    print("=" * 70)
    print(f"TABLE: {table_name}")
    print("=" * 70)

    # Total row count (DuckDB aggregation — no data loaded)
    total = con.execute(f"SELECT count(*) FROM read_parquet('{path}')").fetchone()[0]
    print(f"Total rows: {total:,}")

    # Pull a sample into a Polars DataFrame
    df = con.execute(
        f"SELECT * FROM read_parquet('{path}') LIMIT {SAMPLE}"
    ).pl()

    print(f"\n--- Schema ({len(df.columns)} columns) ---")
    for col, dtype in zip(df.columns, df.dtypes):
        spec = SCHEMAS[table_name].column_map().get(col)
        pk = " [PK]" if spec and spec.is_pk else ""
        print(f"  {col:<28} {str(dtype):<20} {pk}")

    print(f"\n--- head(5) ---")
    display(df.head(5))

    print(f"\n--- describe() (numeric columns) ---")
    num_cols = [c for c, d in zip(df.columns, df.dtypes)
                if d in (pl.Float32, pl.Float64, pl.Int32, pl.Int64, pl.UInt32, pl.UInt64)]
    if num_cols:
        display(df.select(num_cols).describe())

    print(f"\n--- Null rates (sample of {SAMPLE:,}) ---")
    null_rates = {c: round(df[c].null_count() / len(df), 4) for c in df.columns}
    for c, r in sorted(null_rates.items(), key=lambda x: -x[1]):
        bar = "▓" * int(r * 20)
        print(f"  {c:<28} {r:.2%}  {bar}")

    # Timestamp ranges
    ts_cols = [c for c in df.columns if c in ("timestamp", "submit_time")]
    if ts_cols:
        print(f"\n--- Timestamp ranges (full table via DuckDB) ---")
        for tc in ts_cols:
            row = con.execute(
                f"SELECT min({tc}), max({tc}) FROM read_parquet('{path}')"
            ).fetchone()
            span = (row[1] - row[0]) / 86400 if row[0] and row[1] else 0
            print(f"  {tc}: {row[0]} → {row[1]}  ({span:.1f} days)")

    print()


In [ ]:
explore_table("instance_events")

In [ ]:
explore_table("machine_events")

In [ ]:
explore_table("resource_usage")

In [ ]:
explore_table("resource_request")

In [ ]:
explore_table("task_metadata")